# Evaluate BLEU Score

In [ ]:
import os
import torch
import evaluate
from datasets import load_dataset
from tqdm import tqdm


/Users/tonyavis/miniconda3/envs/transformer_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
from utils.data_loader import load_wmt14_en_de, get_training_corpus
from configs.english_german_config import English_german_config
from model.Transformer import Transformer
from model.beam_search import BeamSearch
from model.bpe_tokenizer import build_and_train_BPE_tokenizer
from model.utils import load_checkpoint
from inference import translate_sentence

In [8]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


In [9]:
def load_chpt(cfg:English_german_config, device)->Transformer:
    model = Transformer(cfg)
    chpt_path = os.path.join(cfg.MODEL_DIR, "checkpoints", cfg.checkpoint_name)

    if not os.path.exists(chpt_path):
        raise FileNotFoundError(f"Checkpoint not found at {chpt_path}")
    
    print(f"Loading checkpoint from {chpt_path}...")
    checkpoint = torch.load(chpt_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    return model

In [ ]:
cfg = English_german_config()

# TODO: Update with better model
# NOTE: Config must be the same as the one used to train the this checkpoint!
cfg.checkpoint_name = ""


In [12]:
tokenizer = build_and_train_BPE_tokenizer(
    cfg=cfg, load_wmt14_en_de=load_wmt14_en_de, get_training_corpus=get_training_corpus
)

model = load_chpt(cfg, device)


Loading existing trained Universal BPE Tokenizer from: (/Users/tonyavis/Main/AI_projects_and_res/Transformer/model/saved_models/tokenizer/wmt_14_shared_bpe_tokenizer_universal.json)...
Loading checkpoint from /Users/tonyavis/Main/AI_projects_and_res/Transformer/model/checkpoints/transformer_epoch_1_OVERFIT_TEST_percent_ds.pt...


In [13]:
# Load the BLEU metric
sacreblue = evaluate.load("sacrebleu")

In [14]:
print("Loading validation split of the dataset...")
val_ds = load_dataset("wmt14", "de-en", split="validation")

Loading validation split of the dataset...


In [15]:
# Test on a subset to save time
num_samples = 100
subset = val_ds.select(range(num_samples))

predictions = []
references = []

In [16]:
print(f"Translating {num_samples} sentences for evaluation...")
for i in tqdm(subset):
    eng_text = i["translation"]["en"]
    target_de_text = i["translation"]["de"]

    pred_text = translate_sentence(eng_text, model, tokenizer, cfg, device)

    predictions.append(pred_text)
    references.append([target_de_text])

Translating 100 sentences for evaluation...


100%|██████████| 100/100 [03:27<00:00,  2.07s/it]


In [17]:
print("\nCalculating BLEU score...")
results = sacreblue.compute(predictions=predictions, references=references)
print(f"\n Final BLEU score: {results['score']:.2f}")


Calculating BLEU score...

 Final BLEU score: 0.12
